# Anticipate the next action — an LSTM

**Track C · Egocentric Vision** · maps to lesson C4 (recognition & anticipation).

Anticipation = predict what happens **next**. We generate action sequences from a hidden grammar (a Markov process over a verb vocabulary) and train an LSTM to predict the next action — then measure top-1/top-2 accuracy against the chance baseline.

> CPU is fine.

In [ ]:
import os, torch, torch.nn as nn, matplotlib.pyplot as plt
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 1500)); V, Lseq = 6, 12
verbs = ["take", "wash", "cut", "cook", "pour", "place"]
trans = torch.rand(V, V) + torch.eye(V) * 2.0; trans = trans / trans.sum(1, keepdim=True)   # structured grammar

## 1 · Generate sequences from the hidden grammar

In [ ]:
def batch(n):
    seqs = torch.zeros(n, Lseq, dtype=torch.long)
    for b in range(n):
        s = torch.randint(0, V, (1,)).item()
        for i in range(Lseq):
            seqs[b, i] = s; s = torch.multinomial(trans[s], 1).item()
    return seqs.to(device)
print("example:", [verbs[i] for i in batch(1)[0].tolist()])

## 2 · Model — embedding + LSTM + next-token head

In [ ]:
class Antic(nn.Module):
    def __init__(self):
        super().__init__(); self.emb = nn.Embedding(V, 32); self.lstm = nn.LSTM(32, 64, batch_first=True); self.head = nn.Linear(64, V)
    def forward(self, x): return self.head(self.lstm(self.emb(x))[0])
model = Antic().to(device); opt = torch.optim.Adam(model.parameters(), 3e-3)

## 3 · Train — predict token t+1 from tokens ≤ t

In [ ]:
hist = []
for step in range(STEPS + 1):
    x = batch(128); logits = model(x[:, :-1])
    loss = nn.functional.cross_entropy(logits.reshape(-1, V), x[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0:
        with torch.no_grad():
            xv = batch(512); lg = model(xv[:, :-1]); tgt = xv[:, 1:]
            top1 = (lg.argmax(-1) == tgt).float().mean().item()
            top2 = (lg.topk(2, -1).indices == tgt.unsqueeze(-1)).any(-1).float().mean().item()
        hist.append((step, top1)); print(f"step {step:4d}  top-1 {top1:.3f}  top-2 {top2:.3f}  (chance {1/V:.3f})")

## 4 · Compare — accuracy vs. chance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(*zip(*hist), "-o", label="top-1"); ax.axhline(1 / V, ls="--", c="C7", label="chance")
ax.set_xlabel("step"); ax.set_ylabel("next-action accuracy"); ax.legend(); ax.grid(alpha=.3); ax.set_title("the model learned the grammar")
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/C_action_anticipation_lstm/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/C_action_anticipation_lstm"; os.makedirs(run, exist_ok=True)
torch.save(model.state_dict(), f"{run}/lstm.pt")
json.dump({"top1": hist}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/ropedia-" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Anticipation supports **D** planning and **AG** agents.

### Where to go next
- Feed real video features (from the VideoMAE / CLIP labs) instead of action IDs.
- Predict actions further ahead (anticipation horizon), or the time-to-next-action.